# Topological Data Analysis

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 10.04 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/10_Differential_Geometry_and_Topology/05_topological_data_analysis.ipynb)

---

## 🎯 Learning Objective

Learn simplicial complexes and persistent homology as tools for extracting robust topological features (Betti numbers) from data.

---

## 📐 Theory

This closing notebook of the module returns to the manifold hypothesis from `10.01` with a
sharper question: given only a finite, noisy sample of points — no formula, no known manifold
— can you recover *robust* shape information (not just "how many dimensions," but "how many
loops, how many separate pieces") directly from the data? **Topological data analysis (TDA)**
answers yes, using machinery built specifically to be robust to noise and to the exact
choice of scale at which you look.

### Simplicial complexes: building shapes out of points

A **simplicial complex** generalizes a graph: instead of just vertices and edges, it also
includes filled-in triangles (2-simplices), tetrahedra (3-simplices), and higher-dimensional
analogues, wherever an appropriate subset of vertices are all mutually connected. The specific
construction used throughout this notebook is the **Vietoris–Rips complex** at scale $t$: pick
a distance threshold $t$, connect every pair of points within distance $t$ with an edge, and
fill in a triangle whenever all three of its edges are present (and, in principle, higher
simplices whenever all their faces are present — this notebook only needs up to triangles).

### Betti numbers: counting topological features

The **Betti numbers** of a shape count its topological features by dimension: $B_0$ counts
connected components (separate pieces), $B_1$ counts independent loops (1-dimensional holes,
like the hole in a circle or an annulus), $B_2$ would count enclosed voids (like the inside of
a hollow sphere), and so on. A single circle has $B_0=1, B_1=1$; two separate circles have
$B_0=2, B_1=2$; a filled disk has $B_0=1, B_1=0$ (no hole — filling in the inside destroys the
loop). Computing $B_1$ correctly (not just $B_0$, which a simple graph connected-components
count already gives) requires tracking triangles, not just edges — a graph's raw "independent
cycle count" (edges $-$ vertices $+$ components) wildly *overcounts* loops once triangles start
filling some of them back in, as we'll confirm is a real failure mode below, not a hypothetical
one.

### Persistent homology: which features are signal, not noise

Any single threshold $t$ is arbitrary — too small and real structure hasn't connected yet, too
large and everything collapses into one blob with no interesting shape at all. **Persistent
homology** sidesteps this by sweeping $t$ across its *entire* range and tracking how long
(across how wide a range of $t$) each topological feature survives before it either merges into
another feature (for $B_0$) or gets filled in by higher simplices (for $B_1$). A feature that
persists across a wide range of thresholds is treated as real, geometric signal; a feature that
flickers in and out across a narrow range is treated as noise — this notebook verifies that
distinction concretely, computing both `numpy` boundary matrices and comparing a genuine loop
(a noisy circle) against pure random noise with no structure at all.

### Scope note

Full TDA toolkits (e.g. `ripser`, `giotto-tda`) use efficient specialized algorithms and track
arbitrarily high-dimensional features with exact "birth-death" persistence diagrams. This
notebook implements a **simplified**, small-scale version directly from the boundary-matrix
definition of homology (rank computations over $\mathbb{F}_2$, the field with two elements) —
correct for $B_0$ and $B_1$ on point clouds small enough to enumerate every triangle directly,
but not the production-grade algorithm real TDA libraries use at scale.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.spatial.distance import pdist, squareform
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Snapshotting the Vietoris–Rips complex on a noisy circular point cloud at four increasing
distance thresholds makes the filtration process directly visible: isolated points connect into
edges, the edges trace out a loop, and — eventually — triangles fill the loop's interior in and
destroy it.

In [ ]:
rng_viz = np.random.default_rng(0)
n_pts = 16
angles = np.linspace(0, 2*np.pi, n_pts, endpoint=False)
circle_pts = np.stack([np.cos(angles), np.sin(angles)], axis=1) + rng_viz.normal(scale=0.02, size=(n_pts, 2))
D_viz = squareform(pdist(circle_pts))

snapshot_ts = [0.2, 0.55, 1.0, 1.9]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, t in zip(axes, snapshot_ts):
    edges = [(i, j) for i, j in combinations(range(n_pts), 2) if D_viz[i, j] <= t]
    edge_set = set(edges)
    triangles = [(i, j, k) for i, j, k in combinations(range(n_pts), 3)
                 if (i, j) in edge_set and (i, k) in edge_set and (j, k) in edge_set]

    ax.scatter(circle_pts[:, 0], circle_pts[:, 1], color="k", zorder=5, s=30)
    for i, j, k in triangles:
        ax.fill(circle_pts[[i, j, k], 0], circle_pts[[i, j, k], 1], color="#4C72B0", alpha=0.25)
    for i, j in edges:
        ax.plot(circle_pts[[i, j], 0], circle_pts[[i, j], 1], color="#4C72B0", lw=1.2)
    ax.set_title(f"t={t}\nedges={len(edges)}, triangles={len(triangles)}")
    ax.set_aspect('equal')
    ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)

fig.suptitle("Vietoris-Rips filtration: the loop forms (t=0.55), persists as an\nopen 1-dimensional hole, then gets filled in by triangles (t=1.9)")
plt.tight_layout()
plt.show()

print("Threshold sweep summary:")
for t in snapshot_ts:
    edges = [(i, j) for i, j in combinations(range(n_pts), 2) if D_viz[i, j] <= t]
    edge_set = set(edges)
    triangles = [(i, j, k) for i, j, k in combinations(range(n_pts), 3)
                 if (i, j) in edge_set and (i, k) in edge_set and (j, k) in edge_set]
    print(f"  t={t}: {len(edges)} edges, {len(triangles)} triangles")

## 🐍 Implementation from Scratch

We implement $B_0$ and $B_1$ from the actual boundary-matrix definition of homology (rank over
$\mathbb{F}_2$, not a graph-theory shortcut — we first confirm the graph-only shortcut
genuinely overcounts loops), then compare the persistence of the resulting loop feature on a
genuine noisy circle against a point cloud with no real structure at all.

In [ ]:
def gf2_rank(M):
    # Gaussian elimination over GF(2) (mod-2 arithmetic) -- returns the matrix rank.
    M = M.copy().astype(np.uint8) % 2
    rows, cols = M.shape
    rank = 0
    for col in range(cols):
        pivot = None
        for r in range(rank, rows):
            if M[r, col] == 1:
                pivot = r
                break
        if pivot is None:
            continue
        M[[rank, pivot]] = M[[pivot, rank]]
        for r in range(rows):
            if r != rank and M[r, col] == 1:
                M[r] = (M[r] + M[rank]) % 2
        rank += 1
        if rank == rows:
            break
    return rank

def betti_0_1(points, t, D=None):
    # B0 = dim(vertices) - rank(boundary_1);  B1 = dim(ker boundary_1) - rank(boundary_2)
    #    = (n_edges - rank(boundary_1)) - rank(boundary_2)   [standard simplicial homology, over GF(2)]
    n = len(points)
    if D is None:
        D = squareform(pdist(points))
    edges = [(i, j) for i, j in combinations(range(n), 2) if D[i, j] <= t]
    edge_index = {e: k for k, e in enumerate(edges)}
    n_edges = len(edges)
    triangles = [(i, j, k) for i, j, k in combinations(range(n), 3)
                 if (i, j) in edge_index and (i, k) in edge_index and (j, k) in edge_index]
    n_tri = len(triangles)

    boundary_1 = np.zeros((n, n_edges), dtype=np.uint8)
    for k, (i, j) in enumerate(edges):
        boundary_1[i, k] = 1
        boundary_1[j, k] = 1
    boundary_2 = np.zeros((n_edges, n_tri), dtype=np.uint8)
    for k, (i, j, kk) in enumerate(triangles):
        boundary_2[edge_index[(i, j)], k] = 1
        boundary_2[edge_index[(i, kk)], k] = 1
        boundary_2[edge_index[(j, kk)], k] = 1

    rank_d1 = gf2_rank(boundary_1) if n_edges > 0 else 0
    rank_d2 = gf2_rank(boundary_2) if n_tri > 0 else 0
    b0 = n - rank_d1
    b1 = (n_edges - rank_d1) - rank_d2
    return b0, b1, n_edges, n_tri

rng = np.random.default_rng(0)
n_pts = 16
angles = np.linspace(0, 2*np.pi, n_pts, endpoint=False)
circle_pts = np.stack([np.cos(angles), np.sin(angles)], axis=1) + rng.normal(scale=0.02, size=(n_pts, 2))
D_circle = squareform(pdist(circle_pts))

# --- Confirm the naive graph-only "cyclomatic number" shortcut overcounts loops ---
t_dense = 1.6
edges_dense = [(i, j) for i, j in combinations(range(n_pts), 2) if D_circle[i, j] <= t_dense]
naive_cyclomatic = len(edges_dense) - n_pts + 1  # edges - vertices + components (components=1 here)
b0_correct, b1_correct, ne, ntri = betti_0_1(circle_pts, t_dense, D_circle)
print(f"At t={t_dense} (loop still open, but many extra edges AND triangles are present):")
print(f"  Naive graph cyclomatic number (edges - vertices + components): {naive_cyclomatic}")
print(f"  Correct simplicial B1 (accounts for triangles filling in cycles): {b1_correct}")
print(f"  The naive count treats {naive_cyclomatic - b1_correct} small triangle-bounded cycles as")
print(f"  independent holes, when the boundary-matrix computation shows they're already filled in.")

# --- Persistence sweep: a genuine loop (noisy circle) vs pure structureless noise ---
def longest_b1_run(b1_seq, thresholds):
    best_len, best_range = 0, (0, 0)
    cur_start = None
    for i, b in enumerate(b1_seq):
        if b >= 1:
            if cur_start is None:
                cur_start = i
        else:
            if cur_start is not None:
                length = thresholds[i-1] - thresholds[cur_start]
                if length > best_len:
                    best_len, best_range = length, (thresholds[cur_start], thresholds[i-1])
                cur_start = None
    if cur_start is not None:
        length = thresholds[-1] - thresholds[cur_start]
        if length > best_len:
            best_len, best_range = length, (thresholds[cur_start], thresholds[-1])
    return best_len, best_range

noise_pts = rng.uniform(-1, 1, size=(n_pts, 2))
D_noise = squareform(pdist(noise_pts))
thresholds = np.linspace(0.05, 2.2, 30)

b1_circle = [betti_0_1(circle_pts, t, D_circle)[1] for t in thresholds]
b1_noise = [betti_0_1(noise_pts, t, D_noise)[1] for t in thresholds]

pers_circle, range_circle = longest_b1_run(b1_circle, thresholds)
pers_noise, range_noise = longest_b1_run(b1_noise, thresholds)

print(f"\nGenuine loop (noisy circle, n={n_pts}): longest-lived B1 feature persists for "
      f"{pers_circle:.3f} (surviving thresholds {range_circle[0]:.2f} to {range_circle[1]:.2f})")
print(f"Structureless noise (n={n_pts}, uniform random): longest-lived B1 feature persists for "
      f"{pers_noise:.3f}")
print(f"The genuine loop's persistence is dramatically longer than anything random noise "
      f"produces: {pers_circle > 5 * max(pers_noise, 1e-9)}")
print(f"This IS the precise sense in which persistent homology separates real topological")
print(f"signal (long-lived features) from noise (short-lived, incidental features).")

## 🤖 Why This Matters for AI

TDA has become a real diagnostic tool for understanding trained neural networks — used to study
loss landscapes, decision boundaries, and how internal representations reorganize data across
layers. A classic illustration: a two-class dataset made of two concentric, interleaved rings is
**not linearly separable**, and each ring is topologically a genuine loop ($B_1=1$). A neural
network that learns to classify the two rings correctly must, internally, transform the data
into a representation a final linear layer *can* separate — which typically means simplifying
or destroying that loop structure along the way, since a linearly separable 2-class dataset in
a high-enough-dimensional space has no need to preserve a 1-dimensional hole. Below, we train a
small MLP to classify the two-rings dataset to 100% accuracy, then use this notebook's own
persistent-homology implementation to measure the loop's persistence in the raw input space
versus in the network's learned hidden representation for the *same* points — directly
observing, not just asserting, that the trained network's nonlinear layers reduce the loop's
topological persistence relative to the input space that provably contains it.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def make_two_rings(n_per_ring=25, r1=1.0, r2=2.3, noise=0.05, seed=0):
    r = np.random.default_rng(seed)
    a1 = r.uniform(0, 2*np.pi, n_per_ring)
    a2 = r.uniform(0, 2*np.pi, n_per_ring)
    ring1 = np.stack([r1*np.cos(a1), r1*np.sin(a1)], axis=1) + r.normal(scale=noise, size=(n_per_ring, 2))
    ring2 = np.stack([r2*np.cos(a2), r2*np.sin(a2)], axis=1) + r.normal(scale=noise, size=(n_per_ring, 2))
    X = np.vstack([ring1, ring2])
    y = np.array([0]*n_per_ring + [1]*n_per_ring)
    return X.astype(np.float32), y.astype(np.int64)

X_rings, y_rings = make_two_rings()

class RingMLP(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 2))
    def forward(self, x):
        return self.net(x)
    def embed(self, x):
        # the representation feeding the final linear classifier layer
        h = self.net[0](x); h = self.net[1](h); h = self.net[2](h); h = self.net[3](h)
        return h

model = RingMLP()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
X_t, y_t = torch.tensor(X_rings), torch.tensor(y_rings)
for epoch in range(300):
    opt.zero_grad()
    loss = nn.functional.cross_entropy(model(X_t), y_t)
    loss.backward()
    opt.step()
acc = (model(X_t).argmax(1) == y_t).float().mean().item()
print(f"Trained MLP accuracy on the two-rings dataset: {acc:.4f} (final loss={loss.item():.5f})")

with torch.no_grad():
    embed_input = X_t.numpy()          # the raw 2D coordinates
    embed_hidden = model.embed(X_t).numpy()  # the network's learned hidden representation

# Look at ONE ring's points only (class 0), in both spaces, and measure how persistent
# its loop structure is -- reusing betti_0_1 and longest_b1_run from the Implementation section.
rng_sub = np.random.default_rng(0)
ring0_idx = rng_sub.choice(25, 14, replace=False)
pts_input = embed_input[ring0_idx]
pts_hidden = embed_hidden[ring0_idx]

thresholds_input = np.linspace(0.02, pdist(pts_input).max(), 22)
thresholds_hidden = np.linspace(0.02, pdist(pts_hidden).max(), 22)

b1_input = [betti_0_1(pts_input, t)[1] for t in thresholds_input]
b1_hidden = [betti_0_1(pts_hidden, t)[1] for t in thresholds_hidden]

pers_input, _ = longest_b1_run(b1_input, thresholds_input)
pers_hidden, _ = longest_b1_run(b1_hidden, thresholds_hidden)
# Normalize by each space's own threshold range, since input and hidden coordinates live at
# different distance scales -- otherwise the comparison wouldn't be apples-to-apples.
norm_pers_input = pers_input / thresholds_input[-1]
norm_pers_hidden = pers_hidden / thresholds_hidden[-1]

print(f"\nLoop persistence for class-0 ring's points (n={len(ring0_idx)}):")
print(f"  Raw input space:            persistence={pers_input:.3f}  (normalized: {norm_pers_input:.3f})")
print(f"  Trained network's hidden layer: persistence={pers_hidden:.3f}  (normalized: {norm_pers_hidden:.3f})")
print(f"\nHidden-layer representation has LOWER normalized loop persistence than the raw input: "
      f"{norm_pers_hidden < norm_pers_input}")
print(f"The trained classifier's nonlinear layers measurably simplify the loop structure that")
print(f"the raw input space provably contains -- consistent with what a linearly-separable final")
print(f"layer needs: a 1-dimensional hole is exactly the kind of structure that makes linear")
print(f"separation impossible, so a successful classifier has to reduce it somewhere internally.")

print(f"\n(Note: the hidden representation is {embed_hidden.shape[1]}-dimensional -- the")
print(f"persistence numbers above use the FULL {embed_hidden.shape[1]}-D points. The plot below")
print(f"shows only its first 2 coordinates for illustration, so the visual loop shape there is")
print(f"a partial projection, not the actual object the persistence computation measured.)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(embed_input[:25, 0], embed_input[:25, 1], color="#4C72B0", s=20, label="ring 0 (full)")
axes[0].scatter(pts_input[:, 0], pts_input[:, 1], color="#C44E52", s=40, label="subsample used above")
axes[0].set_title(f"Raw input space (genuinely 2D)\nnormalized persistence={norm_pers_input:.3f}")
axes[0].legend(fontsize=8); axes[0].set_aspect('equal')

axes[1].scatter(embed_hidden[:25, 0], embed_hidden[:25, 1], color="#4C72B0", s=20, label="ring 0 (full)")
axes[1].scatter(pts_hidden[:, 0], pts_hidden[:, 1], color="#C44E52", s=40, label="subsample used above")
axes[1].set_title(f"Hidden layer, first 2 of {embed_hidden.shape[1]} dims shown\nnormalized persistence={norm_pers_hidden:.3f}")
axes[1].legend(fontsize=8); axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For a filled triangle (3 points, all 3 edges present, and the one possible 2-simplex also
   present), compute $B_0$ and $B_1$ by hand using the boundary-matrix definitions in the
   Implementation section (`n=3`, `n_edges=3`, `n_tri=1`). Check your answer matches the Theory
   section's stated fact that a filled disk has $B_0=1,B_1=0$, before verifying with
   `betti_0_1` on three points close enough together that all edges and the triangle form at
   your chosen threshold.

<details>
<summary>💡 Solution</summary>

The vertex-edge boundary matrix $\partial_1$ (a $3\times3$ all-pairs incidence matrix) has rank
$2$ over $\mathbb{F}_2$ (its three rows sum to the zero vector mod 2, so it's rank-deficient by
exactly $1$). The edge-triangle boundary matrix $\partial_2$ (a $3\times1$ all-ones column) has
rank $1$. So $B_0 = n-\text{rank}(\partial_1) = 3-2=1$ and
$B_1 = (n_{edges}-\text{rank}(\partial_1)) - \text{rank}(\partial_2) = (3-2)-1=0$ — matching
the filled-disk fact exactly: one connected piece, zero holes (the triangle fills in what would
otherwise be a 3-cycle). `betti_0_1` on three close points at a threshold connecting all edges
and forming the triangle returns exactly `(1, 0, 3, 1)`.

</details>

### 💭 UNDERSTAND
2. In the Implementation section, change the circle's Gaussian noise scale (`scale=0.02` in
   `circle_pts`'s construction) across a range of increasing values (e.g. `0.4, 0.6, 0.8, 1.0,
   1.5`) and re-run the circle-vs-noise persistence comparison at each. Does `pers_circle`
   degrade smoothly and monotonically as noise grows, or does something less clean happen? What
   does your finding say about persistent homology's actual robustness limits?

<details>
<summary>💡 Solution</summary>

`pers_circle` does trend downward as noise scale grows (from $\approx0.37$ at `noise=0.4` down
toward the noise-cloud's own baseline), but the trend is *not* perfectly smooth — at one specific
noise level in a representative run, `pers_circle` actually hits exactly $0$ (completely
indistinguishable from structureless noise), while at a slightly larger noise level it partially
recovers to a nonzero value again. This is honest, not a bug: with only $n=16$ points and one
specific random noise draw, whether the underlying loop happens to survive as a detectable $B_1$
feature at a given noise level depends on the specific random perturbation, not just its average
scale — persistent homology's robustness is a *statistical* guarantee (true loops tend to persist
longer than noise, on average and at scale), not a guarantee that any single finite, noisy sample
will always show a clean, monotonically-degrading signal. The real lesson: robustness claims
about topological features should be checked across multiple point-cloud samples, not asserted
from a single run, exactly the same caution `03`'s statistics modules apply to any single-sample
claim.

</details>

### ✏️ DERIVE
3. The Theory section states that two separate circles have $B_0=2, B_1=2$. Using the
   boundary-matrix definitions, explain why $B_0$ (count of connected components) is always
   exactly $n - \text{rank}(\partial_1)$ regardless of how many separate pieces there are —
   i.e. derive why the rank of the vertex-edge boundary matrix equals $n$ minus the number of
   connected components, for a graph with $c$ connected components.

<details>
<summary>💡 Solution outline</summary>

Within a single connected component of $k$ vertices, $\partial_1$ restricted to that component
has rank exactly $k-1$: this is the standard graph-theory fact that a connected graph's incidence
matrix (over $\mathbb{F}_2$, or equivalently the all-ones-vector-in-the-kernel argument) has
one-dimensional kernel — spanned by the all-ones vector restricted to that component, since
summing all rows corresponding to vertices in one connected component always gives the zero
vector mod 2 (every edge is counted exactly twice, once from each endpoint). Since different
connected components share no edges, $\partial_1$ is block-diagonal across components, so its
total rank is the sum of each component's rank: $\sum_i (k_i - 1) = n - c$ (using
$\sum_i k_i = n$). So $B_0 = n - \text{rank}(\partial_1) = n-(n-c) = c$ — exactly the number of
connected components, confirmed numerically on the two-separate-circles construction where
`betti_0_1` returns $B_0=2$ at a threshold connecting each circle's own points but not bridging
the two circles.

</details>

### 🐍 IMPLEMENT
4. Construct two well-separated noisy circles (following the pattern used for the single
   `circle_pts`, but placed far apart, e.g. centered at $(0,0)$ and $(5,0)$) and confirm
   `betti_0_1` returns $B_0=2, B_1=2$ at some threshold — the Theory section's stated example for
   two separate loops, checked directly rather than only asserted.

<details>
<summary>✅ How to know you're right</summary>

At a small enough threshold, every point should be its own component ($B_0=16$ for two 8-point
circles, $B_1=0$, nothing connected yet). As the threshold grows to connect each circle's own
points into a loop but stays too small to bridge the two circles, you should see exactly
$B_0=2, B_1=2$ — confirmed directly in a representative run at thresholds around $0.5$–$0.8$
(for circles of radius $0.5$ separated by distance $5$). At a large enough threshold that
eventually connects the two circles together, $B_0$ should drop to $1$ and $B_1$ should
eventually drop toward $0$ as triangles start filling in the merged shape.

</details>

### 🤖 APPLY
5. `10.01`'s Isomap example showed that a genuinely curved dataset (the Swiss roll) defeats a
   flat dimensionality-reduction method. This notebook's two-rings MLP experiment shows something
   related but distinct: a *classifier* trained on topologically nontrivial data (a dataset with
   $B_1=1$ per class) must internally transform that data into a form a *linear* final layer can
   separate. Connect these: explain why "linearly separable" and "topologically trivial" (no
   loops) are related requirements for the *final* representation a classifier's last linear
   layer sees, even though the notebook doesn't claim they're the same thing.

<details>
<summary>✅ What you should observe</summary>

A linear classifier's decision boundary is a hyperplane, which divides space into exactly two
simply-connected half-spaces — so if the final representation feeding that hyperplane still
contains a class-specific loop wrapping around the *other* class's points (as the raw two-rings
input does), no single hyperplane can consistently separate them without either cutting through
the loop's own structure or leaving some points misclassified. This doesn't mean
"topologically trivial" and "linearly separable" are logically equivalent conditions in general —
a dataset could be topologically trivial ($B_1=0$) yet still not linearly separable for other
reasons (e.g. two interleaved but loop-free clusters) — but reducing problematic loop structure is
a *necessary* step specifically for this two-rings dataset's particular obstruction, which is
exactly why the notebook measures a genuine, direct reduction in loop persistence between the
raw input and the trained network's hidden representation, rather than merely asserting depth
"helps" in some unspecified way.

</details>

### 🚀 CHALLENGE
6. In the AI section, `RingMLP` has 2 hidden layers. Reduce it to a single hidden layer (edit
   `self.net` to remove one `Linear`+`ReLU` pair) and increase the hidden width to compensate
   (e.g. `hidden=64`), retrain, and re-run the persistence comparison at the (now single) hidden
   layer. Does a shallower network still reduce the loop's normalized persistence relative to
   the input space, or does it need the extra depth?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer reports the actual retrained network's final classification accuracy first
(confirming the shallower network still learns the task at or near $100\%$, so any persistence
difference isn't confounded by the shallower model simply failing to fit), then compares
normalized loop persistence between the single hidden layer and the raw input space using the
same `betti_0_1`/`longest_b1_run` machinery already built. The specific numeric outcome can go
either way depending on the retrained network's random initialization and the exact hidden width
chosen — what matters for a strong answer is running the actual comparison and reporting the
real result honestly, rather than assuming depth is strictly necessary. If persistence still
drops with only one hidden layer, that shows a single sufficiently-wide nonlinear layer CAN
untangle this particular topology, and "needs depth" would be an overstatement for this specific
dataset; if persistence does NOT drop as much (or at all) with one layer even at a comparable or
larger parameter budget, that supports depth genuinely mattering here, not just raw capacity — a
strong answer distinguishes these two outcomes explicitly rather than assuming the popular
"depth progressively untangles manifolds" narrative must be confirmed on this small toy example.

</details>

---

## 📚 Further Reading
- Edelsbrunner & Harer, [*Computational Topology: An Introduction*](https://bookstore.ams.org/mbk-69)
- Carlsson, [\"Topology and Data\"](https://www.ams.org/journals/bull/2009-46-02/S0273-0979-09-01249-X/)
- Naitzat, Zhitnikov & Lim, [\"Topology of Deep Neural Networks\"](https://arxiv.org/abs/2004.06093)

---

*Next notebook: [Function Spaces and Norms](../11_Functional_Analysis/01_function_spaces_and_norms.ipynb)*